# Week 52, Day 5: Review V: modelling behaviour, and Year 1 in one view

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w052/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: can the parameters be recovered?

Thirty agents with known learning rates and inverse temperatures play 300 trials of a two-armed bandit with a reversal. Write nll(params, a, rw), the negative log-likelihood of a Q-learner's choices, fit every agent with L-BFGS-B, and correlate fitted with true parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
g = np.random.default_rng(50)
T, mu = 300, np.array([0.25, 0.75])
true = np.c_[g.uniform(0.1, 0.6, 30), g.uniform(2, 8, 30)]

def simulate(alpha, beta):
    Q, a, rw = np.zeros(2), np.zeros(T, int), np.zeros(T)
    for t in range(T):
        a[t] = g.random() < 1 / (1 + np.exp(-beta * (Q[1] - Q[0])))
        rw[t] = g.random() < (mu if t < T // 2 else mu[::-1])[a[t]]
        Q[a[t]] += alpha * (rw[t] - Q[a[t]])
    return a, rw

def nll(params, a, rw):
    alpha, beta = params
    Q, L = np.zeros(2), 0.0
    for t in range(T):
        p1 = 1 / (1 + np.exp(-beta * (Q[1] - Q[0])))
        L -= ...
        Q[a[t]] += alpha * (rw[t] - Q[a[t]])
    return L

fitted = np.zeros((30, 2))
for i, (al, be) in enumerate(true):
    a, rw = simulate(al, be)
    fitted[i] = ...
r_alpha = np.corrcoef(true[:, 0], fitted[:, 0])[0, 1]
r_beta = np.corrcoef(true[:, 1], fitted[:, 1])[0, 1]
print(f"recovery: r = {r_alpha:.2f} for alpha, r = {r_beta:.2f} for beta")
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].scatter(true[:, 0], fitted[:, 0]); ax[0].set_xlabel("true alpha"); ax[0].set_ylabel("fitted")
ax[1].scatter(true[:, 1], fitted[:, 1]); ax[1].set_xlabel("true beta"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "fitted holds alpha and beta for every simulated agent, within bounds"
    assert fitted.shape == (30, 2) and np.all(fitted[:, 0] >= 0.01) and np.all(fitted[:, 0] <= 1)
def test_recovery():
    "both parameters are recovered with correlations above 0.6"
    assert r_alpha > 0.6 and r_beta > 0.6

test_fit()
test_recovery()
print("All checks passed")

### Exercise: Problem 2: BIC against cross-validation

For each of 20 simulated datasets, with a true history weight of 0.5 and then 0, fit logistic models with and without the previous choice. Count how often BIC prefers the history model, and how often the history model has the higher log-likelihood on held-out trials (train on the first 600 trials, test on the last 400).

In [ ]:
def bic(ll, k, n):
    return ...

def logreg(X, y):
    w = np.zeros(X.shape[1])
    for _ in range(25):
        p = 1 / (1 + np.exp(-X @ w)); w -= np.linalg.solve(-(X * (p * (1 - p))[:, None]).T @ X - 1e-6 * np.eye(len(w)), X.T @ (y - p))
    return w

def ll(X, y, w):
    z = X @ w; return np.sum(y * z - np.logaddexp(0, z))

agree_bic, agree_cv = {}, {}
for wh in [0.5, 0.0]:
    nb = nc = 0
    for rep in range(20):
        c = g.choice([-1, -0.5, -0.25, 0, 0.25, 0.5, 1], 1000); y = np.zeros(1000); prev = 0.0
        for t in range(1000):
            y[t] = g.random() < 1 / (1 + np.exp(-(3 * c[t] + wh * prev))); prev = 2 * y[t] - 1
        Xf = np.c_[c, np.r_[0, 2 * y[:-1] - 1], np.ones(1000)]; Xs = Xf[:, [0, 2]]
        nb += ...
        tr, te = slice(0, 600), slice(600, 1000)
        nc += ll(Xf[te], y[te], logreg(Xf[tr], y[tr])) > ll(Xs[te], y[te], logreg(Xs[tr], y[tr]))
    agree_bic[wh], agree_cv[wh] = nb, nc
    print(f"true history weight {wh}: BIC prefers history in {nb}/20, cross-validation in {nc}/20")

In [ ]:
import numpy as np
def test_bic():
    "bic(ll, k, n) is k log n minus twice the log-likelihood"
    assert abs(bic(-100.0, 3, 50) - (3 * np.log(50) + 200)) < 1e-12
def test_agree():
    "with a real history effect both criteria prefer the history model in at least 18 of 20 datasets"
    assert agree_bic[0.5] >= 18 and agree_cv[0.5] >= 18
def test_null():
    "with no history effect BIC prefers the simpler model more often than cross-validation"
    assert agree_bic[0.0] <= agree_cv[0.0]

test_bic()
test_agree()
test_null()
print("All checks passed")

### Exercise: Problem 3: from spikes to a psychometric curve

Fifty neurons fire Poisson counts with rates r0 exp(k_i c), half preferring right (k positive) and half left. The ideal observer chooses right when sum_i k_i (n_i - Tw r0) is positive. Simulate 4,000 trials at each contrast, fit a cumulative Gaussian with no lapses to the choices, and compare its sigma with 1 over the square root of the Fisher information at zero contrast.

In [ ]:
from scipy.special import erf
g = np.random.default_rng(51)
N, r0, Tw = 50, 10.0, 0.2
k = np.r_[g.uniform(0.5, 3, N // 2), -g.uniform(0.5, 3, N // 2)]
I0 = ...
contrasts = np.array([-0.3, -0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2, 0.3])
p_right = np.zeros(len(contrasts))
for j, c in enumerate(contrasts):
    n = g.poisson(Tw * r0 * np.exp(k * c), (4000, N))
    p_right[j] = ...
nll_psy = lambda s: -np.sum(4000 * (p_right * np.log(np.clip(0.5 * (1 + erf(contrasts / (s * np.sqrt(2)))), 1e-9, 1))
                                  + (1 - p_right) * np.log(np.clip(0.5 * (1 - erf(contrasts / (s * np.sqrt(2)))), 1e-9, 1))))
sigma_fit = minimize(lambda s: nll_psy(s[0]), [0.1], bounds=[(0.005, 2)]).x[0]
print(f"Fisher information at zero contrast: {I0:.1f}; predicted threshold 1/sqrt(I) = {1 / np.sqrt(I0):.4f}")
print(f"fitted psychometric sigma: {sigma_fit:.4f}")
print("P(right):", np.round(p_right, 3))

In [ ]:
import numpy as np
def test_fisher():
    "I0 is the Fisher information at zero contrast, T r0 times the sum of k squared"
    assert abs(I0 - Tw * r0 * np.sum(k ** 2)) < 1e-9
def test_curve():
    "p_right rises from below 0.1 to above 0.9 across the contrasts"
    assert p_right[0] < 0.1 and p_right[-1] > 0.9
def test_threshold():
    "the fitted psychometric sigma is within 15% of 1 over the square root of I0"
    assert abs(sigma_fit * np.sqrt(I0) - 1) < 0.15

test_fisher()
test_curve()
test_threshold()
print("All checks passed")